# 03 · Forestry — forest segmentation & forest-loss mapping (Amazon + Atlantic Forest)

**CRISP module:** Forestry · **Task:** forest / non-forest segmentation; forest *loss* is the difference between
two dates · **Model:** U-Net / ResNet-34 on Sentinel-2 Red, Green, Blue, NIR

Data: Bragagnolo et al. (2021), Zenodo 4498086, CC BY 4.0 — hand-checked forest masks from two Brazilian biomes.
The official test images are unlabelled, so the official **validation** images are our test set.
Per-image percentile normalisation makes the model independent of how the dataset was scaled to bytes, so it can
run on raw Sentinel-2 L2A imagery at the CRISP forestry sites. Runtime on a T4: about 20 min (+ ~5 min download).

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = "forestry"
from gstp.data import forest as F
from gstp.data.segmentation import SegDataset, split_samples
from gstp.models import build_unet, count_parameters, export_onnx
from gstp import train_seg, baselines, viz
from gstp.utils import save_json
import numpy as np, pandas as pd, torch

DATA = WORK / "data" / "forest"
RESULTS = REPO / "results" / MODULE
CKPT_DIR = DRIVE / MODULE
for p in (DATA, RESULTS / "figures", CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

## 1. Download (2 × 1.2 GB) and extract

In [ ]:
import urllib.request
subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
for name in ("AMAZON.rar", "ATLANTIC FOREST.rar"):
    dst = DATA / name
    if not dst.exists():
        url = "https://zenodo.org/records/4498086/files/" + name.replace(" ", "%20") + "?download=1"
        print("downloading", name); urllib.request.urlretrieve(url, dst)
    marker = DATA / (name + ".done")
    if not marker.exists():
        subprocess.run(["unrar", "x", "-o+", "-inul", str(dst), str(DATA) + "/"], check=True); marker.touch()

In [ ]:
train_all, rep_tr = F.build_samples(DATA, "training")
test, rep_te = F.build_samples(DATA, "validation")
print(rep_tr); print(rep_te)
assert rep_tr["paired"] > 0.95 * rep_tr["images"], "Image/mask pairing incomplete — inspect the folder layout"
assert rep_te["paired"] > 0, "No labelled validation images found — inspect the folder layout"
train, val = split_samples(train_all, 0.12, seed=42)
print(f"train {len(train)} | val {len(val)} | test {len(test)}")
# Visual check that masks line up with images
viz.prediction_grid([{"rgb": viz.rgb(SegDataset([s], "none").load(0)[0], (0, 1, 2)),
                      "mask": SegDataset([s], "none").load(0)[1],
                      "prob": SegDataset([s], "none").load(0)[1].astype(float), "thr": 0.5, "name": s.id}
                     for s in train[:2]], "Pairing check (prediction panel = ground truth)", color=(0.1, 0.8, 0.2));

## 2. Train

In [ ]:
cfg = train_seg.SegTrainConfig(name="forest", in_channels=4, bands=F.BANDS, epochs=60, batch_size=16, crop_size=256,
                               lr=3e-4, patience=12,
                               extra={"norm": "percentile", "dataset": "Amazon + Atlantic Forest (Zenodo 4498086)",
                                      "resolution_m": 10})
tr = SegDataset(train, "percentile", train=True, crop_size=256)
va = SegDataset(val, "percentile")
model = build_unet(4)
summary = train_seg.train(model, tr, va, cfg, CKPT_DIR)
viz.training_curves(summary["history"], title="forest", path=RESULTS / "figures" / "curves_forest.png");

## 3. Test once (per biome), compare with an NDVI threshold

In [ ]:
thr = summary["chosen_threshold"]
report = train_seg.test_report(model, {"test": SegDataset(test, "percentile")}, thr, RESULTS / "test_forest.json",
                               group_key=lambda s: s.meta["biome"])
bl = baselines.tune_and_test(val, {"test": test}, lambda x: baselines.normalized_difference(x[3], x[0]), ">")
save_json({"NDVI threshold": bl}, RESULTS / "baselines.json")
rows = [{"model": "U-Net (R,G,B,NIR)", **report["test"]["at_chosen_threshold"]},
        {"model": f"NDVI > {bl['threshold']:.3f}", **bl["test"]}]
for b, v in report["test"]["per_group"].items():
    rows.append({"model": f"U-Net — {b} only", **v})
table = pd.DataFrame(rows)[["model", "iou", "f1", "precision", "recall", "miou", "accuracy"]].rename(columns={"iou": "IoU (forest)"})
table.to_csv(RESULTS / "results_table.csv", index=False)
table.style.format(precision=3).hide(axis="index")

In [ ]:
ds, raw = SegDataset(test, "percentile"), SegDataset(test, "none")
ex = []
for i in np.linspace(0, len(ds) - 1, 4).astype(int):
    x, y = ds.load(i)
    with torch.no_grad():
        p = torch.sigmoid(model.eval()(torch.from_numpy(x)[None].to(get_device()))).squeeze().cpu().numpy()
    ex.append({"rgb": viz.rgb(raw.load(i)[0], (0, 1, 2)), "mask": y, "prob": p, "thr": thr, "name": ds.samples[i].id})
viz.prediction_grid(ex, "Forest — test", RESULTS / "figures" / "examples_test.png", color=(0.1, 0.8, 0.2))
export_onnx(model, 4, 512, str(CKPT_DIR / "forest.onnx"))
save_json({"summary": {k: v for k, v in summary.items() if k != "history"}, "test": report, "baseline": bl},
          RESULTS / "summary.json")

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))